# Silver - Google Trends 2025

## Objetivo

Transformar a tabela Bronze do Google Trends em uma estrutura analítica
padronizada por data e categoria.

## Transformações

- conversão da data semanal;
- conversão dos índices para tipo numérico;
- transformação de colunas para linhas;
- padronização das categorias;
- filtragem do período de referência;
- inclusão de metadados de processamento.

## Categorias

- Notícias
- Esportes
- Música
- Humor
- Games

In [0]:
from pyspark.sql import functions as F

In [0]:
df_trends_bronze = spark.table(
    "workspace.mvp_bronze.google_trends_2025"
)

print("Linhas Bronze:", df_trends_bronze.count())
print("Colunas Bronze:", len(df_trends_bronze.columns))

display(df_trends_bronze.limit(10))

In [0]:
for coluna in df_trends_bronze.columns:
    print(coluna)

In [0]:
df_trends_base = (
    df_trends_bronze
    .select(
        "time",
        "noticia",
        "esportes",
        "musica",
        "humor",
        "jogo_eletronico"
    )
)

In [0]:
df_trends_tipado = (
    df_trends_base
    .withColumn(
        "data_semana",
        F.to_date("time", "yyyy-MM-dd")
    )
)

In [0]:
df_trends_tipado = (
    df_trends_tipado
    .withColumn(
        "noticia_int",
        F.expr("try_cast(noticia as int)")
    )
    .withColumn(
        "esportes_int",
        F.expr("try_cast(esportes as int)")
    )
    .withColumn(
        "musica_int",
        F.expr("try_cast(musica as int)")
    )
    .withColumn(
        "humor_int",
        F.expr("try_cast(humor as int)")
    )
    .withColumn(
        "games_int",
        F.expr("try_cast(jogo_eletronico as int)")
    )
)

In [0]:
df_trends_long = (
    df_trends_tipado
    .selectExpr(
        "data_semana",
        """
        stack(
            5,
            'Notícias', noticia_int,
            'Esportes', esportes_int,
            'Música', musica_int,
            'Humor', humor_int,
            'Games', games_int
        ) as (categoria, indice_trends)
        """
    )
)

In [0]:
df_trends_long = (
    df_trends_long
    .withColumn(
        "ano_semana",
        F.year("data_semana")
    )
    .withColumn(
        "periodo_referencia",
        F.when(
            F.col("data_semana") >= F.lit("2025-01-01"),
            "2025"
        ).otherwise("Semana inicial de transição")
    )
)

In [0]:
df_trends_silver = (
    df_trends_long
    .withColumn(
        "ano_referencia",
        F.lit(2025)
    )
    .withColumn(
        "pais",
        F.lit("Brazil")
    )
    .withColumn(
        "fonte",
        F.lit("Google Trends")
    )
    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
display(
    df_trends_silver
    .agg(
        F.count("*").alias("total_registros"),

        F.sum(
            F.when(F.col("data_semana").isNull(), 1).otherwise(0)
        ).alias("data_nula"),

        F.sum(
            F.when(F.col("categoria").isNull(), 1).otherwise(0)
        ).alias("categoria_nula"),

        F.sum(
            F.when(F.col("indice_trends").isNull(), 1).otherwise(0)
        ).alias("indice_nulo"),

        F.min("indice_trends").alias("indice_minimo"),
        F.max("indice_trends").alias("indice_maximo")
    )
)

In [0]:
display(
    df_trends_silver
    .groupBy("categoria")
    .agg(
        F.count("*").alias("registros"),
        F.avg("indice_trends").alias("indice_medio")
    )
    .orderBy(F.desc("indice_medio"))
)

In [0]:
display(
    df_trends_silver
    .agg(
        F.min("data_semana").alias("data_minima"),
        F.max("data_semana").alias("data_maxima"),
        F.countDistinct("data_semana").alias("semanas")
    )
)

In [0]:
(
    df_trends_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.mvp_silver.google_trends_2025"
    )
)

In [0]:
%sql

SHOW TABLES IN workspace.mvp_silver;

## Resultado da transformação Silver - Google Trends

A série semanal do Google Trends foi convertida de formato amplo
para formato longo.

A estrutura passou de:

`data | noticias | esportes | musica | humor | games`

para:

`data_semana | categoria | indice_trends`

Os índices foram convertidos para valores numéricos e as categorias
foram padronizadas de acordo com a taxonomia principal do MVP.

A semana iniciada em 29/12/2024 foi preservada por representar
a semana de transição que contém dias de janeiro de 2025.

Tabela criada:

`workspace.mvp_silver.google_trends_2025`